# Parquet to RRD

Minimal chunk/lens path: read the parquet as a lazy chunk stream, derive semantic `Boxes2D` columns from the existing bbox columns, and write an RRD.

In [ ]:
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import rerun as rr
from rerun.experimental import DeriveLens, LazyChunkStream, ParquetReader, Selector

In [ ]:
parquet_path: Path = Path("../data/coco_train/000000.parquet")
if not parquet_path.exists():
    parquet_path = Path("packages/egoexo-forge/data/coco_train/000000.parquet")

parquet_path = parquet_path.resolve()
rrd_path: Path = parquet_path.with_suffix(".bbox.rrd")

parquet_path, rrd_path

In [ ]:
def xywh_array(boxes: pa.Array) -> np.ndarray:
    flattened_boxes: pa.Array = pc.list_flatten(boxes)
    return np.asarray(flattened_boxes.to_numpy(zero_copy_only=False), dtype=np.float32).reshape((-1, 4))


def centers_from_xywh(boxes: pa.Array) -> pa.Array:
    boxes2d = rr.Boxes2D(array=xywh_array(boxes), array_format=rr.Box2DFormat.XYWH)
    return boxes2d.centers.as_arrow_array()


def half_sizes_from_xywh(boxes: pa.Array) -> pa.Array:
    boxes2d = rr.Boxes2D(array=xywh_array(boxes), array_format=rr.Box2DFormat.XYWH)
    return boxes2d.half_sizes.as_arrow_array()


source_stream: LazyChunkStream = ParquetReader(
    parquet_path,
    entity_path_prefix="/source/sam3d_body/parquet",
    column_grouping="individual",
).stream()

bbox_columns: list[str] = [
    "/source/sam3d_body/parquet/bbox",
]
bbox_column_stream: LazyChunkStream = source_stream.filter(content=bbox_columns)

bbox_stream: LazyChunkStream = bbox_column_stream.lenses(
    [
        DeriveLens("bbox", output_entity="/world/cam/pinhole/pred/bbox").to_component(
            rr.Boxes2D.descriptor_centers(),
            Selector(".").pipe(centers_from_xywh),
        ),
        DeriveLens("bbox", output_entity="/world/cam/pinhole/pred/bbox").to_component(
            rr.Boxes2D.descriptor_half_sizes(),
            Selector(".").pipe(half_sizes_from_xywh),
        ),
    ],
    output_mode="drop_unmatched",
)

bbox_stream.write_rrd(
    rrd_path,
    application_id="egoexo_forge_parquet_notebook",
    recording_id=parquet_path.stem,
)

rrd_path